In [ ]:
# ============
# ENV SWITCH
# ============
RUN_ENV = "kaggle_tpu"

# Keep within single-session TPU. Adjust later once it runs end-to-end.
PROFILES = {
    "kaggle_tpu": dict(
        platform="tpu",
        # GRPO is slow per-step; start conservative and scale later
        max_steps=200,
        eval_every=20,
        save_every=50,
        # sequence controls (keep output under <1k tokens overall)
        max_prompt_length=256,
        total_generation_steps=512,
        # training batch-ish controls (will be set in GRPO config)
        num_generations=4,
    ),
}

CFG = PROFILES[RUN_ENV]
CFG


In [ ]:
import os

# Let JAX auto-detect TPU on Kaggle TPU runtime
os.environ.pop("JAX_PLATFORMS", None)

# Reduce log noise
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# TPU-friendly allocator behavior
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")


In [ ]:
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Tunix (PyPI recommended) + common deps
pip_install([
    "google-tunix[prod]",
    "datasets",
    "transformers",
    "huggingface_hub",
    "grain",
    "tensorflow",
    "tensorflow_datasets",
    "sentencepiece",
    "tensorboardX",
    "ipywidgets",
])

# Qwix is used for LoRA in the official Tunix Gemma GRPO demo
pip_install(["git+https://github.com/google/qwix"])


In [ ]:
import os
import re
import json
from pathlib import Path

import jax
import optax
from orbax import checkpoint as ocp

from huggingface_hub import snapshot_download
from flax import nnx

import qwix

# Tunix: generation + tokenizer adapter + gemma3 + RL
from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib

from tunix.models.gemma3 import model as gemma_lib
from tunix.models.gemma3 import params_safetensors as params_safetensors_lib

from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner
from tunix.rl.rollout import base_rollout

from tunix.sft import metrics_logger


In [ ]:
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("WORK_DIR:", WORK_DIR)
print("Devices:", jax.devices())


In [ ]:
# Strict required output format:
# <reasoning>...</reasoning>
# <answer>...</answer>

RE_EXACT = re.compile(
    r"^<reasoning>\s*(.*?)\s*</reasoning>\s*<answer>\s*(.*?)\s*</answer>\s*$",
    re.DOTALL,
)

def match_format_exactly(prompt: str, completion: str, **kwargs) -> float:
    return 1.0 if RE_EXACT.match(completion) else -1.0

def match_format_approximately(prompt: str, completion: str, **kwargs) -> float:
    has_reasoning = ("<reasoning>" in completion) and ("</reasoning>" in completion)
    has_answer = ("<answer>" in completion) and ("</answer>" in completion)
    return 0.5 if (has_reasoning and has_answer) else -0.5


In [ ]:
# ===== Model =====
MODEL_ID = "google/gemma-3-1b-it"   # safest for single-session TPU; faster than 2B
# (Gemma2 2B is possible, but start with 1B to get a working pipeline)

# ===== Generation during GRPO training =====
MAX_PROMPT_LENGTH = CFG["max_prompt_length"]
TOTAL_GENERATION_STEPS = CFG["total_generation_steps"]
TEMPERATURE = 0.9
TOP_P = 1.0
TOP_K = 50

# ===== GRPO =====
NUM_GENERATIONS = CFG["num_generations"]
NUM_ITERATIONS = CFG["max_steps"]   # map training steps to iterations
BETA = 0.04
EPSILON = 0.2

# ===== LoRA (lightweight, good for single-session) =====
RANK = 16
ALPHA = 16


In [ ]:
SYSTEM_INSTRUCTION = (
    "You must respond in the following format exactly:\n"
    "<reasoning>...</reasoning>\n"
    "<answer>...</answer>\n"
    "Use English only. Do not use tools. Keep the full output under 1000 tokens."
)

def make_prompt(q: str) -> str:
    return f"{SYSTEM_INSTRUCTION}\n\nUser: {q}\nAssistant:"

train_prompts = [
    make_prompt("What is 17 + 25?"),
    make_prompt("Explain why the sky looks blue in one paragraph."),
    make_prompt("Summarize: 'Cats sleep a lot but can be playful.'"),
    make_prompt("Write a short idea for a sci-fi story about a lost satellite."),
]

# GRPO expects iterable[dict] with key "prompts"
def prompt_iter(prompts):
    while True:
        for p in prompts:
            yield {"prompts": [p]}

train_dataset = prompt_iter(train_prompts)
val_dataset = prompt_iter(train_prompts[:2])


In [ ]:
# ===== Model + tokenizer (HF gated repo auth) =====
import os
import json
import jax
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

# NOTE: tokenizer_lib, gemma_lib, params_safetensors_lib, qwix, RANK, ALPHA
# must already be imported/defined in earlier cells.

MODEL_ID = "google/gemma-3-1b-it"

# Pull Hugging Face token from Kaggle Secrets (name must be HF_TOKEN)
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# Make token available to HF hub internals
os.environ["HF_TOKEN"] = HF_TOKEN

# Download model snapshot (requires access accepted on HF)
ignore_patterns = ["*.pth"]
local_model_path = snapshot_download(
    repo_id=MODEL_ID,
    ignore_patterns=ignore_patterns,
    token=HF_TOKEN,
)
print("Model downloaded to:", local_model_path)

# Detect EOS tokens (optional)
EOS_TOKENS = []
gen_cfg_path = os.path.join(local_model_path, "generation_config.json")
if os.path.exists(gen_cfg_path):
    with open(gen_cfg_path, "r") as f:
        gen_cfg = json.load(f)
    EOS_TOKENS = gen_cfg.get("eos_token_id", [])
if isinstance(EOS_TOKENS, int):
    EOS_TOKENS = [EOS_TOKENS]
print("EOS token IDs:", EOS_TOKENS)

# Tokenizer adapter (required for RLCluster/sampler)
hf_tokenizer = AutoTokenizer.from_pretrained(local_model_path, use_fast=True)
if hf_tokenizer.pad_token is None:
    hf_tokenizer.pad_token = hf_tokenizer.eos_token

tokenizer = tokenizer_lib.TokenizerAdapter(hf_tokenizer)
print("Tokenizer loaded:", type(hf_tokenizer), "pad_token:", hf_tokenizer.pad_token)

# ===== Sharding mesh =====
devices = jax.devices()
num_tpus = len(devices)
print("TPU devices:", num_tpus)

# Critical fix:
# Gemma3 loader shards some tensors on 'tp' along a dimension of size 4,
# so tp must divide 4. On an 8-core TPU slice, use (fsdp=2, tp=4).
if num_tpus == 8:
    mesh_shape = (2, 4)          # fsdp=2, tp=4
elif num_tpus == 4:
    mesh_shape = (1, 4)          # fsdp=1, tp=4
elif num_tpus == 2:
    mesh_shape = (1, 2)          # fsdp=1, tp=2
elif num_tpus == 1:
    mesh_shape = (1, 1)          # fsdp=1, tp=1
else:
    # Fallback: prefer tp=4 if possible, else tp=1
    if num_tpus % 4 == 0:
        mesh_shape = (num_tpus // 4, 4)
    else:
        mesh_shape = (num_tpus, 1)

mesh = jax.make_mesh(mesh_shape, ("fsdp", "tp"))
print("Mesh:", mesh)

# ===== Base model =====
model_config = gemma_lib.ModelConfig.gemma3_1b()

with mesh:
    base_model = params_safetensors_lib.create_model_from_safe_tensors(
        local_model_path, model_config, mesh
    )

# ===== Apply LoRA to create policy model =====
lora_provider = qwix.LoraProvider(
    module_path=(
        ".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj|"
        ".*attn_vec_einsum"
    ),
    rank=RANK,
    alpha=ALPHA,
)

model_input = base_model.get_model_input()
policy_model = qwix.apply_lora_to_model(base_model, lora_provider, **model_input)
reference_model = base_model

print("Loaded policy (LoRA) + reference models.")


In [ ]:
# ===== Optimizer + logging + checkpointing =====
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

CKPT_DIR = str(WORK_DIR / "ckpts")
TB_DIR = str(WORK_DIR / "tensorboard" / "grpo")

checkpointing_options = ocp.CheckpointManagerOptions(
    save_interval_steps=CFG["save_every"],
    max_to_keep=4,
)

metrics_logging_options = metrics_logger.MetricsLoggerOptions(
    log_dir=TB_DIR,
    flush_every_n_steps=20,
)

optimizer = optax.adamw(learning_rate=3e-6, b1=0.9, b2=0.99, weight_decay=0.1)

# ===== Cluster config (this is the critical fix) =====
cluster_config = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: mesh,
        rl_cluster_lib.Role.REFERENCE: mesh,
        rl_cluster_lib.Role.ROLLOUT: mesh,
    },
    rollout_engine="vanilla",
    offload_to_cpu=False,
    training_config=rl_cluster_lib.RLTrainingConfig(
        actor_optimizer=optimizer,
        eval_every_n_steps=CFG["eval_every"],
        max_steps=CFG["max_steps"],
        mini_batch_size=1,
        train_micro_batch_size=1,
        metrics_logging_options=metrics_logging_options,
        checkpoint_root_directory=CKPT_DIR,
        checkpointing_options=checkpointing_options,
    ),
    rollout_config=base_rollout.RolloutConfig(
        max_tokens_to_generate=CFG["total_generation_steps"],
        max_prompt_length=CFG["max_prompt_length"],
        kv_cache_size=CFG["max_prompt_length"] + CFG["total_generation_steps"] + 256,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        eos_tokens=EOS_TOKENS,
    ),
)

grpo_config = GRPOConfig(
    num_generations=CFG["num_generations"],
    num_iterations=1,
    beta=BETA,
    epsilon=EPSILON,
)

rl_cluster = rl_cluster_lib.RLCluster(
    actor=policy_model,
    reference=reference_model,
    tokenizer=tokenizer,
    cluster_config=cluster_config,
)

trainer = GRPOLearner(
    rl_cluster=rl_cluster,
    reward_fns=[match_format_exactly, match_format_approximately],
    config=grpo_config,
)

with mesh:
    trainer.train(train_dataset, val_dataset)

print("Training finished.")
print("Checkpoints (if any) under:", CKPT_DIR)
